# Group 3 — v3 Embeddings (Reconstruction Loss, No-Log Target)

In [1]:
import sys
import os
from pathlib import Path

def find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for path in [start, *start.parents]:
        if (path / 'src').exists() and (path / 'requirements.txt').exists():
            return path
    raise FileNotFoundError('Project root not found.')


PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd

from src.models.fix_embeddings import FixedGNNConfig, build_fixed_pooled_dataset
from src.models.embeddings import Node2VecConfig

PROJECT_ROOT = find_project_root()
TARGET_DIR   = PROJECT_ROOT / 'src' / 'datasets' / 'targets'
OUTPUT_DIR   = PROJECT_ROOT / 'src' / 'data' / 'embeddings'

pd.set_option('display.max_columns', 200)
print(f'Project root: {PROJECT_ROOT}')

Project root: C:\Users\ruben\Desktop\Universidade\Nova IMS\Tese\Thesis


## Configuration

In [6]:
TARGET_COL           = 'systemic_risk_label'
INCLUDE_RAW_FEATURES = False
YEARS                = range(2016, 2024)
QUARTERS             = (1, 2, 3, 4)

cfg_graphsage_32  = FixedGNNConfig(hidden_dims=(256, 32),  dropout=0.3, lr=0.01, epochs=100, reconstruction_weight=1.0, link_weight=0.0, aggregation='mean', device='cpu')
cfg_graphsage_64  = FixedGNNConfig(hidden_dims=(256, 64),  dropout=0.3, lr=0.01, epochs=100, reconstruction_weight=1.0, link_weight=0.0, aggregation='mean', device='cpu')
cfg_graphsage_128 = FixedGNNConfig(hidden_dims=(256, 128), dropout=0.3, lr=0.01, epochs=100, reconstruction_weight=1.0, link_weight=0.0, aggregation='mean', device='cpu')

cfg_node2vec_32   = Node2VecConfig(embedding_dim=32,  walk_length=20, context_size=10, walks_per_node=10, num_negative_samples=1, batch_size=128, lr=0.01, epochs=100, device='cpu')
cfg_node2vec_64   = Node2VecConfig(embedding_dim=64,  walk_length=20, context_size=10, walks_per_node=10, num_negative_samples=1, batch_size=128, lr=0.01, epochs=100, device='cpu')
cfg_node2vec_128  = Node2VecConfig(embedding_dim=128, walk_length=20, context_size=10, walks_per_node=10, num_negative_samples=5, batch_size=128, lr=0.01, epochs=100, device='cpu')

OUTPUTS = {
    'graphsage_32':  OUTPUT_DIR / 'graphsage_v3_32_srisk_nolog_dataset.parquet',
    'graphsage_64':  OUTPUT_DIR / 'graphsage_v3_64_srisk_nolog_dataset.parquet',
    'graphsage_128': OUTPUT_DIR / 'graphsage_v3_128_srisk_nolog_dataset.parquet',
    'node2vec_32':   OUTPUT_DIR / 'node2vec_v3_32_srisk_nolog_dataset.parquet',
    'node2vec_64':   OUTPUT_DIR / 'node2vec_v3_64_srisk_nolog_dataset.parquet',
    'node2vec_128':  OUTPUT_DIR / 'node2vec_v3_128_srisk_nolog_dataset.parquet',
}

OUTPUTS

{'graphsage_32': WindowsPath('C:/Users/ruben/Desktop/Universidade/Nova IMS/Tese/Thesis/src/data/embeddings/graphsage_v3_32_srisk_nolog_dataset.parquet'),
 'graphsage_64': WindowsPath('C:/Users/ruben/Desktop/Universidade/Nova IMS/Tese/Thesis/src/data/embeddings/graphsage_v3_64_srisk_nolog_dataset.parquet'),
 'graphsage_128': WindowsPath('C:/Users/ruben/Desktop/Universidade/Nova IMS/Tese/Thesis/src/data/embeddings/graphsage_v3_128_srisk_nolog_dataset.parquet'),
 'node2vec_32': WindowsPath('C:/Users/ruben/Desktop/Universidade/Nova IMS/Tese/Thesis/src/data/embeddings/node2vec_v3_32_srisk_nolog_dataset.parquet'),
 'node2vec_64': WindowsPath('C:/Users/ruben/Desktop/Universidade/Nova IMS/Tese/Thesis/src/data/embeddings/node2vec_v3_64_srisk_nolog_dataset.parquet'),
 'node2vec_128': WindowsPath('C:/Users/ruben/Desktop/Universidade/Nova IMS/Tese/Thesis/src/data/embeddings/node2vec_v3_128_srisk_nolog_dataset.parquet')}

## GraphSAGE v3

In [3]:
for cfg, key in [
    (cfg_graphsage_32,  'graphsage_32'),
    (cfg_graphsage_64,  'graphsage_64'),
    (cfg_graphsage_128, 'graphsage_128'),
]:
    df = build_fixed_pooled_dataset(
        config=cfg,
        years=YEARS,
        quarters=QUARTERS,
        target_col=TARGET_COL,
        include_raw_features=INCLUDE_RAW_FEATURES,
        target_dir=TARGET_DIR,
        output_path=OUTPUTS[key],
    )
    emb_cols = [c for c in df.columns if c.startswith('emb_')]
    print(f'{key:15s}  shape={df.shape}  emb_cols={len(emb_cols)}')

Loaded 2016 Q1: 4548 banks, 11631 edges
Loaded 2016 Q2: 4548 banks, 11632 edges
Loaded 2016 Q3: 4548 banks, 11937 edges
Loaded 2016 Q4: 4548 banks, 11938 edges
Loaded 2017 Q1: 4548 banks, 11939 edges
Loaded 2017 Q2: 4548 banks, 11940 edges
Loaded 2017 Q3: 4548 banks, 11941 edges
Loaded 2017 Q4: 4548 banks, 11981 edges
Loaded 2018 Q1: 4548 banks, 12416 edges
Loaded 2018 Q2: 4548 banks, 12417 edges
Loaded 2018 Q3: 4548 banks, 12418 edges
Loaded 2018 Q4: 4548 banks, 12419 edges
Loaded 2019 Q1: 4548 banks, 12420 edges
Loaded 2019 Q2: 4548 banks, 12421 edges
Loaded 2019 Q3: 4548 banks, 12422 edges
Loaded 2019 Q4: 4548 banks, 12423 edges
Loaded 2020 Q1: 4548 banks, 12424 edges
Loaded 2020 Q2: 4548 banks, 12451 edges
Loaded 2020 Q3: 4548 banks, 12452 edges
Loaded 2020 Q4: 4548 banks, 12453 edges
Loaded 2021 Q1: 4548 banks, 12454 edges
Loaded 2021 Q2: 4548 banks, 12455 edges
Loaded 2021 Q3: 4548 banks, 12456 edges
Loaded 2021 Q4: 4548 banks, 12457 edges
Loaded 2022 Q1: 4548 banks, 12458 edges


## Node2Vec v3

In [4]:
for cfg, key in [
    (cfg_node2vec_32,  'node2vec_32'),
    (cfg_node2vec_64,  'node2vec_64'),
    (cfg_node2vec_128, 'node2vec_128'),
]:
    df = build_fixed_pooled_dataset(
        config=cfg,
        years=YEARS,
        quarters=QUARTERS,
        target_col=TARGET_COL,
        include_raw_features=INCLUDE_RAW_FEATURES,
        target_dir=TARGET_DIR,
        output_path=OUTPUTS[key],
    )
    emb_cols = [c for c in df.columns if c.startswith('emb_')]
    print(f'{key:15s}  shape={df.shape}  emb_cols={len(emb_cols)}')

Loaded 2016 Q1: 4548 banks, 11631 edges
Loaded 2016 Q2: 4548 banks, 11632 edges
Loaded 2016 Q3: 4548 banks, 11937 edges
Loaded 2016 Q4: 4548 banks, 11938 edges
Loaded 2017 Q1: 4548 banks, 11939 edges
Loaded 2017 Q2: 4548 banks, 11940 edges
Loaded 2017 Q3: 4548 banks, 11941 edges
Loaded 2017 Q4: 4548 banks, 11981 edges
Loaded 2018 Q1: 4548 banks, 12416 edges
Loaded 2018 Q2: 4548 banks, 12417 edges
Loaded 2018 Q3: 4548 banks, 12418 edges
Loaded 2018 Q4: 4548 banks, 12419 edges
Loaded 2019 Q1: 4548 banks, 12420 edges
Loaded 2019 Q2: 4548 banks, 12421 edges
Loaded 2019 Q3: 4548 banks, 12422 edges
Loaded 2019 Q4: 4548 banks, 12423 edges
Loaded 2020 Q1: 4548 banks, 12424 edges
Loaded 2020 Q2: 4548 banks, 12451 edges
Loaded 2020 Q3: 4548 banks, 12452 edges
Loaded 2020 Q4: 4548 banks, 12453 edges
Loaded 2021 Q1: 4548 banks, 12454 edges
Loaded 2021 Q2: 4548 banks, 12455 edges
Loaded 2021 Q3: 4548 banks, 12456 edges
Loaded 2021 Q4: 4548 banks, 12457 edges
Loaded 2022 Q1: 4548 banks, 12458 edges


## Output

In [7]:
for name, path in OUTPUTS.items():
    print(f'{name:20s} -> {path.name}')

graphsage_32         -> graphsage_v3_32_srisk_nolog_dataset.parquet
graphsage_64         -> graphsage_v3_64_srisk_nolog_dataset.parquet
graphsage_128        -> graphsage_v3_128_srisk_nolog_dataset.parquet
node2vec_32          -> node2vec_v3_32_srisk_nolog_dataset.parquet
node2vec_64          -> node2vec_v3_64_srisk_nolog_dataset.parquet
node2vec_128         -> node2vec_v3_128_srisk_nolog_dataset.parquet
